# Executed Pipeline Evidence

This notebook reads tracked artifacts produced by the Lab 04 pipeline. Its code is read-only, so it can be executed safely without live infrastructure.

In [1]:
from pathlib import Path
import json

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / 'data/processed/discovered_files.json').exists())
print(f'Artifact root: {PROJECT_ROOT.name}')

Artifact root: Streaming


## Repository discovery

In [2]:
manifest = json.loads((PROJECT_ROOT / 'data/processed/discovered_files.json').read_text())
print('repo_path:', manifest['repo_path'])
print('total_files:', manifest['total_files'])
print('first_5_files:')
for file_path in manifest['files'][:5]:
    print('-', file_path)

repo_path: data/raw/accelerate
total_files: 121
first_5_files:
- benchmarks/big_model_inference/big_model_inference.py
- benchmarks/big_model_inference/measures_util.py
- benchmarks/fp8/ms_amp/ddp.py
- benchmarks/fp8/ms_amp/distrib_deepspeed.py
- benchmarks/fp8/ms_amp/fp8_utils.py


## Kafka topic/event samples

In [3]:
sample_dir = PROJECT_ROOT / 'evidence/kafka'
for name in ['nodes_sample.json', 'edges_sample.json', 'metadata_sample.json', 'errors_sample.json']:
    event = json.loads((sample_dir / name).read_text())
    identity = next((event[key] for key in ('node_id', 'edge_id', 'metadata_id') if key in event), '<error-event>')
    print(f"{name}: schema={event.get('schema_version')} time={event.get('event_time')} id={identity}")

nodes_sample.json: schema=1.0 time=2026-07-04T11:01:47.041898Z id=3ae824950f95ebadd259eb6c27eaaa2278e64ac8e6e376c504042e54b165557c
edges_sample.json: schema=1.0 time=2026-07-04T11:01:47.041898Z id=feee0f76b36c1cba5188e2cafab3b9bd967b88b798cbb59cb09e548af3801285
metadata_sample.json: schema=1.0 time=2026-07-04T11:01:47.041898Z id=59f203db77b71174cb16b9dba979e76280d162b3718b611ee05a2ef28bc561ea
errors_sample.json: schema=1.0 time=2026-07-04T11:01:47.360504Z id=<error-event>


## MongoDB metadata verification

In [4]:
def matching_lines(path, terms, limit=14):
    lines = path.read_text(errors='replace').splitlines()
    selected = [line.strip() for line in lines if any(term.lower() in line.lower() for term in terms)]
    return selected[-limit:]

pipeline_log = PROJECT_ROOT / 'evidence/logs/terminal_2_pipeline_latest.log'
for line in matching_lines(pipeline_log, ['MongoDB', 'metadata', 'duplicate', 'Count delta']):
    print(line)

'src/accelerate/_lab_replay_probe.py', 'metadata_id':
INFO     Count delta: nodes=+3 edges=+7 metadata_documents=+0
INFO     Duplicate count: metadata_id=0 repo/file=0
INFO     MongoDB metadata reflects the replayed file hash.
[07/04/26 14:24:02] INFO     MongoDB metadata documents: 121
INFO     Duplicate metadata_id groups: 0
INFO     Duplicate repo/file groups: 0
INFO     Target metadata: {'event_time':
'src/accelerate/_lab_replay_probe.py', 'metadata_id':
INFO     Duplicate node IDs: 0
INFO     Duplicate edge IDs: 0
│ ❱  55 │   │   nodes, edges, metadata = build_event │
│ ❱ 19 │   nodes, ast_edges, metadata = parse_python │
metadata_id=59f203db77b71174cb16b9dba979e76280d162b371


## Neo4j graph verification

In [5]:
for line in matching_lines(pipeline_log, ['Neo4j totals', 'Duplicate node', 'Duplicate edge', 'placeholder', 'Target file']):
    print(line)

[07/04/26 14:23:50] INFO     Neo4j totals: nodes=114782 edges=319302
INFO     Duplicate node IDs: 0
INFO     Duplicate edge IDs: 0
INFO     Unresolved placeholder nodes: 0
INFO     Target file src/accelerate/_lab_replay_probe.py:
[07/04/26 14:24:03] INFO     Neo4j totals: nodes=114785 edges=319309
INFO     Duplicate node IDs: 0
INFO     Duplicate edge IDs: 0
INFO     Unresolved placeholder nodes: 0
INFO     Target file src/accelerate/_lab_replay_probe.py:


## Replay/checkpoint evidence

In [6]:
checkpoint_log = PROJECT_ROOT / 'evidence/logs/checkpoint_resume.log'
print(checkpoint_log.read_text().strip())
print('--- replay summary ---')
for line in matching_lines(pipeline_log, ['Modified:', 'Count delta:', 'reflects the replayed file hash']):
    print(line)

checkpoint_location=outputs/checkpoints/mongodb_metadata
checkpoint_exists=True
checkpoint_artifacts_before=5
checkpoint_artifacts_after=5
metadata_count_before=121
metadata_count_after=121
result=PASSED checkpoint resumed without duplicating unchanged metadata
--- replay summary ---
INFO     Modified: False
INFO     Modified: True
INFO     Count delta: nodes=+3 edges=+7 metadata_documents=+0
INFO     MongoDB metadata reflects the replayed file hash.
INFO     Modified: True


## Reflection

**What worked:** Stable event identities, connector-backed metadata upsert, checkpoint recovery, and duplicate verification are represented by reproducible artifacts.

**What failed:** Documentation previously drifted from the checked-in discovery manifest, and the book had no executed notebook evidence.

**How it was fixed:** The manifest description was synchronized and this read-only notebook was executed against tracked evidence.

**Remaining limitations:** CFG, DFG, and CALL construction remains lightweight. Modified-file graph replay uses a controlled direct Neo4j cleanup before updated events are republished; metadata replay remains event-driven through Kafka, checkpointed Spark Structured Streaming, and MongoDB.